In [2]:
%pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------- ----------------------- 4.7/11.3 MB 39.9 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 33.9 MB/s  0:00:00

   ---------------------------------------- 0/5 [wrapt]
   ---------------------------------------- 0/5 [wrapt]
   ---------------------------------------- 0/5 [wrapt]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   -------- ------------------------------- 1/5 [patsy]
   ---------------- ----------------------- 2/5 [interface-meta]
   ---------------- ----------------------- 2/5 [interface-meta]
   ------------------------ --------------- 3/5 [formulaic]
   ----------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from data_loading import load_clean

df = load_clean()
df.shape

(668897, 13)

In [2]:
df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount
66,139.0,1,0.75,F,7,1,61,50,B12,'Regular',27000,R11,303.00
93,190.0,1,0.14,B,12,5,50,60,B12,'Diesel',56,R25,1981.84
199,414.0,1,0.14,E,4,0,36,85,B12,'Regular',4792,R11,1456.55
205,424.0,2,0.62,F,10,0,51,100,B12,'Regular',27000,R11,10834.00
223,463.0,1,0.31,A,5,0,45,50,B12,'Regular',12,R73,3986.67


In [3]:
df["age_group"] = pd.cut(df["DrivAge"], bins=[17, 25, 35, 45, 55, 65, 100])

modele_freq = smf.glm(
    "ClaimNb ~ BonusMalus + C(age_group)",
    data=df,
    family=sm.families.Poisson(),
    offset=np.log(df["Exposure"])
).fit()

print(modele_freq.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               668897
Model:                            GLM   Df Residuals:                   668890
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0801e+05
Date:                Thu, 05 Nov 2026   Deviance:                   1.6526e+05
Time:                        13:23:07   Pearson chi2:                 1.17e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.008753
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [9]:
pearson_ratio = (
    modele_freq.pearson_chi2 / modele_freq.df_resid
)

deviance_ratio = (
    modele_freq.deviance / modele_freq.null_deviance
)

print("Rapport de Pearson :", pearson_ratio)
print("Rapport de déviance :", deviance_ratio)

Rapport de Pearson : 1.7510552143244247
Rapport de déviance : 0.965637103426539


In [7]:
np.exp(modele_freq.params)

Intercept                                            0.012623
C(age_group)[T.Interval(25, 35, closed='right')]     0.882826
C(age_group)[T.Interval(35, 45, closed='right')]     1.165986
C(age_group)[T.Interval(45, 55, closed='right')]     1.325992
C(age_group)[T.Interval(55, 65, closed='right')]     1.166788
C(age_group)[T.Interval(65, 100, closed='right')]    1.124260
BonusMalus                                           1.027593
dtype: float64

In [8]:
print(modele_freq.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               668897
Model:                            GLM   Df Residuals:                   668890
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.0801e+05
Date:                Thu, 05 Nov 2026   Deviance:                   1.6526e+05
Time:                        13:23:40   Pearson chi2:                 1.17e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.008753
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [11]:
modele_freq_q = smf.glm(
    "ClaimNb ~ BonusMalus + C(age_group)",
    data=df,
    family=sm.families.Poisson(),
    offset=np.log(df["Exposure"])
).fit(scale="X2")

print(modele_freq_q.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               668897
Model:                            GLM   Df Residuals:                   668890
Model Family:                 Poisson   Df Model:                            6
Link Function:                    Log   Scale:                          1.7511
Method:                          IRLS   Log-Likelihood:                -61686.
Date:                Thu, 05 Nov 2026   Deviance:                   1.6526e+05
Time:                        13:29:55   Pearson chi2:                 1.17e+06
No. Iterations:                     9   Pseudo R-squ. (CS):           0.005008
Covariance Type:            nonrobust                                         
                                                        coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------

In [12]:
df["bm_group"] = pd.cut(df["BonusMalus"], bins=[49, 60, 80, 100, 120, 150, 250])

modele_freq2 = smf.glm(
    "ClaimNb ~ C(bm_group) + C(age_group)",
    data=df,
    family=sm.families.Poisson(),
    offset=np.log(df["Exposure"])
).fit()

print(modele_freq.deviance, modele_freq2.deviance)
np.exp(modele_freq2.params)

165261.05561975756 165006.78989846236


Intercept                                             0.050715
C(bm_group)[T.Interval(60, 80, closed='right')]       2.250219
C(bm_group)[T.Interval(80, 100, closed='right')]      2.917527
C(bm_group)[T.Interval(100, 120, closed='right')]     6.262659
C(bm_group)[T.Interval(120, 150, closed='right')]     8.872192
C(bm_group)[T.Interval(150, 250, closed='right')]    10.908046
C(age_group)[T.Interval(25, 35, closed='right')]      0.815778
C(age_group)[T.Interval(35, 45, closed='right')]      1.102422
C(age_group)[T.Interval(45, 55, closed='right')]      1.263936
C(age_group)[T.Interval(55, 65, closed='right')]      1.111628
C(age_group)[T.Interval(65, 100, closed='right')]     1.072728
dtype: float64

On obtient effectivement une déviance plus faible pour ce nouveau modèle mais ce modèle a plus de paramètres (5 pour le bonus-malus tandis qu'on en avait 1 pour modele_freq) donc la comparaison n'est pas équitable. on va regarder l'AIC qui elle pénalise le nombre de paramètres

In [ ]:
print(modele_freq.aic, modele_freq2.aic)